In [ ]:
import numpy as np
import numba as nb
import scipy.stats as stats
import torch
import matplotlib.pyplot as plt
import pickle
import yaml

from sbi import utils as utils
from sbi import analysis as analysis
from sbi.utils.user_input_checks import process_simulator
from sbi.inference.potentials import ratio_estimator_based_potential
from sbi.inference.posteriors import MCMCPosterior
from sbi.simulators.simutils import simulate_in_batches

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parent if Path.cwd().name == "analysis" else Path.cwd().resolve()
src_dir = project_root / "src"
if str(src_dir) not in sys.path:
    sys.path.append(str(src_dir))

from simulator import smFRET_simulator_wrapper_from_config
from priors import generate_posterior_prior_from_config
from utils import eval_spline_from_config

In [ ]:
def bin_along_time(x, n_bins=100):
    n = len(x)
    n_per_bin = int(n / n_bins)
    x_binned = np.zeros(n_bins)
    for i in range(n_bins):
        x_binned[i] = np.sum(x[i*n_per_bin:(i+1)*n_per_bin])
    return x_binned

In [ ]:
data_cfg = "/home/dingeldein/Desktop/fret_sbi/smFRET_sbi/outputs/2026-03-27/10-00-42/data_config.yaml"
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
prior = generate_posterior_prior_from_config(data_cfg, device=device)
eval_spline = eval_spline_from_config(data_cfg)

In [ ]:
fret_simulator = smFRET_simulator_wrapper_from_config(str(data_cfg))

In [ ]:
true_params = torch.tensor([ 0.5,  2.2121, 0.434, 1.2529, -0.3520,  10.000, 2.00, 0.0100,  0.01000,  0.9500,  0.9500,  0.9000,  0.9000,  0.1000,  0.1000], dtype=torch.float64)

In [ ]:
true_params[5]

In [ ]:
x, y = eval_spline(
    params=true_params,
    x_eval=np.linspace(0, 20, 1000),
)
plt.plot(x, y)

plt.ylim(-5, 5)

In [ ]:
from multiprocessing import Pool
import os

def _init_worker():
    np.random.seed(os.getpid())  # unique seed per worker so x_init differs

with Pool(processes=24, initializer=_init_worker) as pool:
    results = pool.map(fret_simulator, [true_params] * 3)

observation = torch.stack(results)

In [ ]:
idx = 0
plt.figure(figsize=(7, 2.5))
plt.plot(bin_along_time(observation[idx, :, 0].numpy(), n_bins=150), label="Donor")
plt.plot(bin_along_time(observation[idx, :, 1].numpy(), n_bins=150), label="acceptor")
plt.ylabel("Photon count")
plt.xlabel("Time")


In [ ]:
with open("/home/dingeldein/Desktop/fret_sbi/smFRET_sbi/outputs/2026-03-27/10-35-17/last_estimator.pkl", "rb") as f:
    ratio_estimator = pickle.load(f)

In [ ]:
potential, tf = ratio_estimator_based_potential(ratio_estimator, prior, observation.cuda())
posterior = MCMCPosterior(
    potential,
    proposal=prior,
    theta_transform=tf,
    num_chains=10,
    thin=2,
    method="slice_np_vectorized",
)

In [ ]:
import types
import math

# Pre-compute the x_o embedding once — avoids re-running embedding_net_x at every MCMC step
with torch.no_grad():
    x_o_embedded = ratio_estimator.embedding_net_x(
        observation.cuda().reshape(-1, *ratio_estimator.x_shape)
    )  # shape: (n_iid, embed_dim)

ratio_estimator._x_cached_embedded = x_o_embedded

# Save original method so we can fall back to it when shapes don't align
# (e.g. during init strategy where theta arrives in unexpected batch shapes)
_original_combine = ratio_estimator.__class__.combine_theta_and_x

def _fast_combine(self, theta, x):
    """combine_theta_and_x with cached x embedding.

    Falls back to the original (full re-embedding) when the theta tensor's
    element count isn't divisible by prod(theta_shape), which can happen
    during the init strategy phase of MCMC.
    """
    theta_shape_size = math.prod(self.theta_shape)
    n_iid = self._x_cached_embedded.shape[0]

    if theta.numel() % theta_shape_size != 0:
        # Shape mismatch — fall back to original (re-embeds x from scratch)
        return _original_combine(self, theta, x)

    n = theta.numel() // theta_shape_size
    if n % n_iid != 0:
        return _original_combine(self, theta, x)

    # Fast path: use cached x embedding
    # SBI repeats x as AABBCC (repeat_interleave) and theta as ABCABC,
    # so we must mirror that pattern for the cached embedding.
    embedded_theta = self.embedding_net_theta(theta.reshape(n, *self.theta_shape))
    embedded_x = self._x_cached_embedded.repeat_interleave(n // n_iid, dim=0)
    prefix_shape = theta.shape[: -len(self.theta_shape)]
    return torch.cat([embedded_theta, embedded_x], dim=-1).reshape(*prefix_shape, -1)

ratio_estimator.combine_theta_and_x = types.MethodType(_fast_combine, ratio_estimator)

potential, tf = ratio_estimator_based_potential(ratio_estimator, prior, observation.cuda())
posterior = MCMCPosterior(
    potential,
    proposal=prior,
    theta_transform=tf,
    num_chains=10,
    thin=2,
    method="slice_np_vectorized",
)

In [ ]:
samples = posterior.sample(
    (5000,), 
    x=observation.cuda(),
    init_strategy_parameters={'num_candidate_samples': 1000},
).cpu()

In [ ]:
map_estimate = posterior.map(
    num_iter=1000,
    num_init_samples=100,
    show_progress_bars=True,
    force_update=True
)

In [ ]:
_ = analysis.pairplot(
    samples,
    figsize=(7,7),
    points=true_params, 
    points_offdiag={'markersize': 6},
    points_colors='r', 
)

In [ ]:
samples[:, 1:] = samples[:, 1:] - samples[:, 1:].mean(dim=1, keepdim=True)
map_estimate[1:] = map_estimate[1:] - map_estimate[1:].mean()
true_params[1:5] = true_params[1:5] - true_params[1:5].mean()

In [ ]:
for i in range(1000):
    x, y = eval_spline(
        params=samples[i],
        x_eval=np.linspace(0, 20, 1000),
    )
    plt.plot(x, y, color='gray', alpha=0.01)
x, y = eval_spline(
    params=map_estimate[0].cpu(),
    x_eval=np.linspace(0, 20, 1000),
)
plt.plot(x, y, color='black', label='MAP Estimate')
x, y = eval_spline(
    params=true_params,
    x_eval=np.linspace(0, 20, 1000),
)
plt.plot(x, y+0., color='r', label='Ground Truth')
plt.xlabel("Distance (nm)", fontsize=16)
plt.ylabel(r"$\Delta G [k_{B}T]$", fontsize=16)
plt.ylim(-5, 5)
plt.grid()
plt.legend(fontsize=13)